In [1]:
# El Work Folder se presupone /home3/carlos.arenas

# v007_pmu7-fn0k-fc0-ww3_soft-qcd_none_1_1e4.hepmc
# v009_pb-pb-pmu0.0018-fn0k-fc0-ww3_inelastic_none_1_1e6.hepmc

with open("/home3/carlos.arenas/Proyecto/data/v009_pb-pb-pmu0.0018-fn0k-fc0-ww3_inelastic_none_1_1e6.hepmc", "r") as file:
    lines = file.readlines()
    print(len(lines))


10084067


In [2]:
# Split into different events:

event_lines = []
for i, line in enumerate(lines):
    values = line.split(" ")

    if values[0] == "E":
        event_id = values[1]

        event_lines.append([event_id, i])

event_lines.append([0, len(lines)])

print(f"Hay {len(event_lines) - 1} eventos")

Hay 172 eventos


In [4]:
pdg_id_dicc = {13: r"$\mu^-$", 111: r"$\pi^0$", 321: r"$K^+$", 2112: "n", 2212: r"$p^+$"}

def text2list(text):

    # Elimina los espacios blancos.
    import re
    text = re.sub(r"\s+", "", text)

    text = text[1:-1]
    l = text.split(",")

    if l == ['']:
        # print(f"Este venia vacio: {text}")
        return []
    
    try:
        l = [int(x) for x in l]
    except:
        print(f"Exception ocurred with: {l}")
        return None
    
    return l

def for_all_events(func):
    def looped_func():
        for i in range(len(event_lines) - 1):
            event_id = event_lines[i][0]
            start_line = event_lines[i][1]
            end_line = event_lines[i+1][1]

            func(event_id, start_line, end_line)
        return

    return looped_func

def for_single_event(func):
    def func_event1():
        event_nr = 50
        event_id = event_lines[event_nr][0]
        start_line = event_lines[event_nr][1]
        end_line = event_lines[event_nr+1][1]
        func(event_id, start_line, end_line)
        return
    return func_event1

In [37]:
@for_all_events
def count_vertices(event_id, start_line, end_line):
    vertices_counter = 0
    for i in range(end_line - start_line):
        line = lines[start_line + i]
        values = line.split(" ")

        if values[0] == "V":
            vertices_counter += 1
            # print(line)

    print(f"El evento {event_id} tiene {vertices_counter} vértices")

# Esto probablemente debería ser sustituido por uso de diccionarios.
def retrieve(id_, l):
    for x in l:
        if x[0] == id_:
            return x[1]

    return None

def fill_dicc(event_id, start_line, end_line):
    global vertices_particles
    global particles_vertex
    global particles_pid
    global particles_nhit

    vertices_particles.clear()
    particles_vertex.clear()
    particles_pid.clear()
    particles_nhit.clear()

    vertices_particles[0] = []

    for i in range(end_line - start_line):
        line = lines[start_line + i]
        values = line.split(" ")

        if values[0] == "V":
            vertex_id = int(values[1])
            vertex_particles = text2list(values[3])
            
            # vertices_particles.append([vertex_id, vertex_particles])
            vertices_particles[vertex_id] = vertex_particles

        if values[0] == "P":
            particle_id = int(values[1])
            vertex_id = int(values[2])
            pdg_id = int(values[3])

            particles_vertex[particle_id] = vertex_id
            particles_pid[particle_id] = pdg_id

        if values[0] == "A":
            if values[2] == "RawBankLinks":
                particle_id = int(values[1])
                particle_banks = values[3:-1]
                particle_nhit = len(particle_banks)

                particles_nhit[particle_id] = particle_nhit

def fill_directed_vertices_links():
    global directed_vertices_links
    global birth_death_dicc

    directed_vertices_links.clear()
    birth_death_dicc.clear()

    # Con sets ya no me tengo que preocupar de que haya duplicados
    directed_vertices_links = {vertex: set() for vertex in vertices_particles}
    directed_vertices_links[0] = set()

    for vertex, particles in vertices_particles.items():
        for particle in particles:
            mother_vertex = particles_vertex[particle]

            directed_vertices_links[mother_vertex].add(vertex)

            birth_death_dicc[particle] = [mother_vertex, vertex]

    for particle, vertex in particles_vertex.items():
        if not particle in birth_death_dicc.keys():
            birth_death_dicc[particle] = [vertex, None]


In [38]:
def search_mother_vertices():
    # Por ahora, mother_vertices va acumulando las madres de todas las cadenas (ergo de todos los eventos)

    global vertices_particles
    global particles_vertex

    global mother_vertices

    particles = list(particles_vertex.keys())

    for particle in particles:
        for i in range(50):
            mother_vertex = particles_vertex[particle]

            try:
                particle = vertices_particles[mother_vertex][0]
            except:
                if mother_vertex != None:
                    if mother_vertex not in mother_vertices:
                        mother_vertices.append(mother_vertex)
                break

In [39]:
def fill_vertices_output():
    global mother_vertices

    global vertices_output

    vertices_output.clear()

    vertices_output = {vertex: [] for vertex in vertices_particles}

    for vertex in mother_vertices:
        vertices_output[vertex] = []

    for particle, vertex in particles_vertex.items():
        vertices_output[vertex].append(particle)

def find_mother_particles_pid():
    global mother_vertices
    global particles_pid

    mother_particles = [vertices_output[vertex] for vertex in mother_vertices]
    mother_particles_pid = [particles_pid[particle] for particle in mother_particles]

    return mother_particles_pid


In [ ]:
def for_all_vertices_in_chain(upon_visite_func):
    def looped_func(obj, starting_vertex = 0):
        global directed_vertices_links

        visited_vertices = {vertex: 0 for vertex in vertices_particles}
        visited_vertices[0] = 0

        def visitar(obj, vertex):
            # print("Se esta visitando algo")
            visited_vertices[vertex] = 1
            current_chain.append(vertex)

            mod_obj, skip = upon_visite_func(obj, vertex)
            if skip: return

            for daughter_vertex in directed_vertices_links[vertex]:
                if visited_vertices[daughter_vertex] == 0:
                    visitar(obj, daughter_vertex)

        visitar(obj, starting_vertex)

    return looped_func

class counter():
    def __init__(self):
        self.value = 0

@for_all_vertices_in_chain
def sum_hits(hit_counter, this_vertex):
    # print("Se esta haciendo algo")
    for particle in vertices_output[this_vertex]:
        # print("Se esta haciendo algo")
        try:
            nhit = particles_nhit[particle]
            hit_counter.value += nhit
        except KeyError:
            pass

    skip = False
    return hit_counter, skip

@for_all_vertices_in_chain
def count_KS_hits(KS_hits, current_vertex):

    skip = False
    for particle in vertices_particles[current_vertex]:
        # print("Se esta haciendo algo")
        if particles_pid[particle] == 2212:
            # print("Se encontró kei short")
            skip = True

            hit_counter = counter()
            sum_hits(hit_counter, current_vertex)
            KS_hits.append(hit_counter.value) # Igual, problema de scope

    return KS_hits, skip

@for_all_vertices_in_chain
def gritar(obj, vertex):
    print("AHHHH")
    return obj, False

@for_all_vertices_in_chain
def count_KS(KS_counter, vertex):
    skip = False
    for particle in vertices_particles[vertex]:
        # print("Se esta haciendo algo")
        if particles_pid[particle] == 2212:
            skip = True

            KS_counter.value += 1
            
    return KS_counter, skip

In [126]:
vertices_particles = {}
particles_vertex = {}
particles_pid = {}
particles_nhit = {}

# flat_vertex_links = []
directed_vertices_links = {}
birth_death_dicc = {}

mother_vertices = []

vertices_input = []
vertices_output = {}

current_chain = []
chains = []
current_chain_hits = 0
chains_hits = []

KS_hits = []
KS_count = 0

@for_single_event
def main(event_id, start_line, end_line):

    fill_dicc(event_id, start_line, end_line)

    search_mother_vertices()

    fill_directed_vertices_links()
    fill_vertices_output()

    # print(find_mother_particles_pid())

    # count_hits_per_chain()
    hit_counter = counter()
    sum_hits(hit_counter, 0)
    print(hit_counter.value)
    chains_hits.append(hit_counter.value)
    chains.append(current_chain)

    global KS_hits
    count_KS_hits(KS_hits, 0)

    KS_counter = counter()
    count_KS(KS_counter, 0)
    print(f"Hay: {KS_counter.value} K shorts")

    print([particles_pid[particle] for particle in vertices_output[0]])

    print(f"El evento {event_id} tiene {len(vertices_particles)} vértices y {chains_hits[-1]} hits")


In [127]:
main()
print(chains_hits)

print(KS_hits)


12
Hay: 177 K shorts
[1000822080, 1000822080]
El evento 54851502 tiene 22474 vértices y 12 hits
[12]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [93]:
print(KS_count)

0


In [ ]:
s = 0
for particle in particles_nhit:
    print(particle)
    s += particles_nhit[particle]

print(s)

ss = 0
for vertex, particles in vertices_output.items():
    for particle in particles:
        # print(particle)

        try:
            ss += particles_nhit[particle]
        except KeyError:
            pass

print(ss)

print([particles_pid[particle] for particle in vertices_output[0]])

print(directed_vertices_links)

65292
12
12
[1000822080, 1000822080]
{-15983: [-1, -3, -4, -7, -8, -12, -13, -15, -20, -22, -24, -26, -10322, -31, -32, -34, -35, -39, -41, -43, -44, -47, -52, -54, -55, -58, -65, -67, -68, -71, -73, -75, -78, -80, -83, -84, -88, -90, -93, -94, -96, -99, -100, -104, -107, -109, -110, -112, -114, -116, -118, -120, -123, -125, -133, -136, -137, -139, -142, -144, -150, -152, -154, -159, -162, -19670, -168, -171, -174, -19589, -7568, -180, -183, -184, -186, -188, -192, -194, -195, -1157, -200, -204, -207, -209, -215, -233, -235, -3264, -241, -242, -246, -252, -254, -256, -13767, -260, -261, -263, -265, -267, -269, -272, -274, -276, -278, -282, -283, -285, -287, -288, -290, -293, -297, -2868, -300, -301, -304, -307, -308, -311, -313, -314, -316, -6984, -321, -322, -335, -338, -340, -345, -346, -349, -367, -370, -371, -374, -375, -378, -379, -380, -382, -388, -15040, -391, -395, -396, -400, -401, -418, -4836, -13375, -507, -514, -539, -644, -4255, -712, -743, -5562, -15413, -1121, -1567, -68

In [ ]:
for c in chains:
    print(len(c))

print(mother_vertices)

mother_particles = [vertices_output[vertex] for vertex in mother_vertices]
mother_particles = [item for sublist in mother_particles for item in sublist]
# mother_particles = sum(mother_particles, [])

mother_particles_pid = [particles_pid[particle] for particle in mother_particles]
print(mother_particles_pid)

weird_particle = mother_particles[0]

second_vertex = retrieve(0, flat_vertex_links)
b = vertices_output[second_vertex]

e = [particles_pid[x] for x in b]
print(e)


[0]
[90, 90]
[-11]


In [ ]:
different_pids = []
for x in particles_pid:
    if x[1] not in different_pids:
        different_pids.append(x[1])
        print(x[1])

2112
223
211
-213
111
331
-211
113
-311
313
-3122
2224
213
1
21
2
2212
311
-1114
2114
221
321
-323
323
-2112
-2212
-3224
3214
3114
-313
-321
-1
9902210
990
-2114
1114
-2214
-3212
3222
-3222
-2224
-2
-3214
2214
333
3122
-523
-3112
3324
3
3212
-423
-3
3224
-3312
3112
-421
4
3322
423
433
-3114
-3322
3312
-413
-433
-411
421
-4
-3334
413
-3314
3314
411
-3324
431
22
511
-431
310
-11
11
130
14
-13
-16
15
-12
-14
13
20213
12
-20213
10551
20313
10323
10441
-521
-10311
16
1000822080
1000180419
1000260479
90


In [ ]:
# Esta es la segunda copia, dedicada a sumar hits.
chains_hits = []

# Esto podría hacerse más facil con las claves de vertices_particles.
visited_vertices = {}
vertices_neighbours = {}
for x in flat_vertex_links:

    visited_vertices[x[0]] = 0
    vertices_neighbours[x[0]] = []
    visited_vertices[x[1]] = 0
    vertices_neighbours[x[1]] = []

for link in flat_vertex_links:
    # print(link[0])

    vertices_neighbours[link[0]].append(link[1])
    vertices_neighbours[link[1]].append(link[0])

# Otro algoritmo de DFS

chains = []
def visitar(vertex):
    visited_vertices[vertex] = 1
    chains[-1].append(vertex)

    for particle in retrieve(vertex, vertices_output):
        nhit = retrieve(particle, particles_nhit)
        if nhit != None:
            chains_hits[-1] += nhit

    # print(vertices_neighbours[vertex])

    for neigh_vertex in vertices_neighbours[vertex]:
        if visited_vertices[neigh_vertex] == 0:
            visitar(neigh_vertex)

for vertex in visited_vertices.keys():
    if visited_vertices[vertex] == 0:
        chains.append([])

        chains_hits.append(0)
        
        visitar(vertex)

[[0, -15983, -62, -11488, -19255, -13133, -19257, -63, -716, -717, -1697, -8806, -123, -12, -1, -2, -17, -4837, -9905, -12619, -15579, -19873, -19794, -18, -19797, -4501, -6417, -19795, -10008, -15580, -15696, -19796, -19922, -19, -798, -153, -16, -139, -21712, -10498, -21710, -11982, -11981, -16671, -140, -10499, -10500, -11936, -141, -20316, -10517, -19046, -18947, -18945, -11407, -18885, -10518, -13624, -18622, -10519, -18581, -10520, -10521, -20320, -1264, -10947, -142, -13241, -12761, -1231, -22421, -290, -23, -285, -20868, -51, -479, -480, -481, -482, -10070, -136, -15, -130, -9398, -9400, -14581, -131, -9405, -9408, -14229, -132, -9453, -14545, -20934, -340, -24, -321, -322, -13733, -5560, -20918, -22432, -4775, -3727, -87, -11, -80, -21572, -3056, -3057, -3058, -3059, -3060, -3062, -3065, -3066, -3068, -3071, -3072, -3073, -3075, -3076, -3078, -3079, -3081, -3082, -3083, -3085, -3086, -3089, -3090, -3092, -3094, -3095, -3097, -3098, -3100, -3101, -2547, -25, -343, -20938, -38, 

In [ ]:
# print(len(flat_vertex_links))
# print(flat_vertex_links)

# Esto podría hacerse más facil con las claves de vertices_particles.
visited_vertices = {}
vertices_neighbours = {}
for x in flat_vertex_links:

    visited_vertices[x[0]] = 0
    vertices_neighbours[x[0]] = []
    visited_vertices[x[1]] = 0
    vertices_neighbours[x[1]] = []

for link in flat_vertex_links:
    # print(link[0])

    vertices_neighbours[link[0]].append(link[1])
    vertices_neighbours[link[1]].append(link[0])

# Otro algoritmo de DFS

chains = []
def visitar(vertex):
    visited_vertices[vertex] = 1
    chains[-1].append(vertex)

    # print(vertices_neighbours[vertex])

    for neigh_vertex in vertices_neighbours[vertex]:
        if visited_vertices[neigh_vertex] == 0:
            visitar(neigh_vertex)

for vertex in visited_vertices.keys():
    if visited_vertices[vertex] == 0:
        chains.append([])
        visitar(vertex)

print(chains)
for c in chains:
    print(len(c))

[[0, -13337, -9547, -929, -930, -9173, -9179, -9181, -177, -5888, -7954, -5479, -10559, -7793, -7632, -8031, -11647, -178, -5233, -5234, -11202, -12417, -12418, -495, -496, -510, -511, -512, -513, -526, -527, -535, -582, -593, -594, -670, -671, -672, -771, -772, -773, -777, -1023, -7148, -208, -7167, -4378, -9826, -207, -7130, -2406, -2407, -2415, -2416, -2417, -4570, -4581, -13883, -2441, -2442, -2443, -4652, -7102, -2478, -2450, -2451, -2452, -2460, -2469, -2470, -7333, -237, -13, -1, -2, -115, -135, -5, -144, -13558, -55, -1468, -13551, -57, -1529, -9575, -1689, -13473, -62, -2125, -13684, -5024, -5025, -8706, -299, -8627, -8767, -7644, -261, -11306, -5987, -202, -5729, -2395, -2396, -2397, -2398, -10874, -10945, -12841, -12842, -12843, -13204, -6, -145, -4753, -7915, -137, -13773, -471, -18, -14035, -7216, -7547, -9611, -12765, -12766, -445, -12354, -446, -13891, -14134, -447, -448, -449, -450, -451, -452, -453, -454, -455, -456, -457, -12709, -13335, -13942, -13943, -13969, -13982

In [ ]:
print(len(visited_vertices))
print(visited_vertices)

print(f"Hay {len(flat_vertex_links)} links entre vértices")

21080
{0: 0, -20270: 0, -20879: 0, -1: 0, -2: 0, -3: 0, -4: 0, -5: 0, -6: 0, -7: 0, -8: 0, -9: 0, -10: 0, -11: 0, -12: 0, -13: 0, -153: 0, -12550: 0, -14: 0, -15: 0, -16: 0, -17: 0, -18: 0, -19: 0, -20: 0, -21: 0, -22: 0, -23: 0, -24: 0, -25: 0, -26: 0, -27: 0, -28: 0, -29: 0, -30: 0, -31: 0, -32: 0, -33: 0, -34: 0, -90: 0, -1996: 0, -35: 0, -36: 0, -37: 0, -2024: 0, -38: 0, -39: 0, -40: 0, -2034: 0, -41: 0, -42: 0, -43: 0, -44: 0, -45: 0, -46: 0, -134: 0, -10437: 0, -47: 0, -48: 0, -49: 0, -50: 0, -92: 0, -2221: 0, -51: 0, -177: 0, -16212: 0, -52: 0, -53: 0, -54: 0, -55: 0, -56: 0, -57: 0, -58: 0, -59: 0, -60: 0, -19848: 0, -61: 0, -62: 0, -63: 0, -105: 0, -5851: 0, -64: 0, -65: 0, -126: 0, -15912: 0, -66: 0, -67: 0, -68: 0, -69: 0, -70: 0, -112: 0, -6720: 0, -71: 0, -72: 0, -73: 0, -74: 0, -75: 0, -76: 0, -77: 0, -78: 0, -79: 0, -80: 0, -81: 0, -6420: 0, -82: 0, -83: 0, -84: 0, -85: 0, -86: 0, -87: 0, -88: 0, -89: 0, -144: 0, -11444: 0, -91: 0, -93: 0, -94: 0, -95: 0, -96: 0, -97: 0,

In [ ]:
# Creo lista de vecinos de cada vértices

# vertices_neighbours = {x[0]: [] for x in vertices_particles}
# print(vertices_particles)
# print(vertices_neighbours)
for link in flat_vertex_links:
    # print(link[0])
    vertices_neighbours[link[0]].append(x[1])
    vertices_neighbours[link[1]].append(x[0])

# print(vertices_neighbours)

for vertex in visited_vertices.keys():
    print(vertex)
    print(visited_vertices[vertex])
    pass


0
1
-20270
1
-20879
1
-1
1
-2
1
-3
1
-4
1
-5
1
-6
1
-7
1
-8
1
-9
1
-10
1
-11
1
-12
1
-13
1
-153
1
-12550
1
-14
1
-15
1
-16
1
-17
1
-18
1
-19
1
-20
1
-21
1
-22
1
-23
1
-24
1
-25
1
-26
1
-27
1
-28
1
-29
1
-30
1
-31
1
-32
1
-33
1
-34
1
-90
1
-1996
1
-35
1
-36
1
-37
1
-2024
1
-38
1
-39
1
-40
1
-2034
1
-41
1
-42
1
-43
1
-44
1
-45
1
-46
1
-134
1
-10437
1
-47
1
-48
1
-49
1
-50
1
-92
1
-2221
1
-51
1
-177
1
-16212
1
-52
1
-53
1
-54
1
-55
1
-56
1
-57
1
-58
1
-59
1
-60
1
-19848
1
-61
1
-62
1
-63
1
-105
1
-5851
1
-64
1
-65
1
-126
1
-15912
1
-66
1
-67
1
-68
1
-69
1
-70
1
-112
1
-6720
1
-71
1
-72
1
-73
1
-74
1
-75
1
-76
1
-77
1
-78
1
-79
1
-80
1
-81
1
-6420
1
-82
1
-83
1
-84
1
-85
1
-86
1
-87
1
-88
1
-89
1
-144
1
-11444
1
-91
1
-93
1
-94
1
-95
1
-96
1
-97
1
-98
1
-99
1
-100
1
-101
1
-102
1
-103
1
-104
1
-106
1
-107
1
-108
1
-109
1
-110
1
-111
1
-113
1
-4793
1
-114
1
-115
1
-116
1
-117
1
-118
1
-119
1
-120
1
-121
1
-122
1
-123
1
-124
1
-125
1
-127
1
-128
1
-129
1
-130
1
-131
1
-132
1
-133
1
-189
1
-1

In [ ]:
# count_vertices()

# print(vertex_links)
print(len(vertices_particles))
print(len(particles_vertex))

for i in range(100):
    print(vertex_links[i])
    print(f"input particles: {vertices_input[i]}")
    print(f"output particles: {vertices_output[i]}")


21080
60967
[[0, -20270]]
input particles: [-20270, [60764]]
output particles: [-20270, [1, 116, 229, 232, 235, 237, 391, 412, 416, 546, 648, 794, 899, 918, 958, 978, 1024, 1042, 1052, 1062, 1152, 1191, 1355, 1536, 1633, 1661, 1711, 1715, 1745, 1848, 1882, 1996, 2224, 2294, 2344, 2543, 2825, 3016, 3075, 3122, 3306, 3408, 3433, 3573, 3598, 3790, 3940, 4087, 4135, 4238, 4366, 4411, 4527, 4788, 4961, 5048, 5146, 5149, 5157, 5159, 5163, 5401, 5578, 5684, 5754, 5757, 5822, 5971, 6094, 6156, 6204, 6207, 6209, 6302, 6586, 6631, 6829, 6885, 7068, 7205, 7270, 7558, 7572, 7697, 7778, 7858, 7997, 8141, 8145, 8234, 8351, 8443, 8693, 8740, 8826, 8841, 9077, 9125, 9190, 9269, 9340, 9465, 9517, 9526, 9665, 9717, 9804, 9902, 9975, 9977, 9995, 10023, 10144, 10298, 10376, 10464, 10494, 10519, 10626, 10704, 10772, 10791, 10883, 11011, 11055, 11093, 11113, 11173, 11447, 11884, 12377, 14824, 14890, 16483, 19592, 19656, 20160, 21735, 22716, 23098, 28530, 30753, 31185, 32175, 34442, 34508, 39142, 39764, 4191

In [1]:
for line in lines[:100]:
    values = line.split(" ")

    if values[0] == "V":
        vertex_particles = text2list(values[3])

        if values[3] == "":
            print(line)

        if len(vertex_particles) > 2:
            print(line)

    if values[0] == "P":
        print(line)

NameError: name 'lines' is not defined